# Bike Rental Forecasting**Project 26 of 50** — Time Series Forecasting Portfolio

## Project OverviewForecast hourly bike rental demand using the Kaggle Bike Sharing Demand dataset.Classic Kaggle competition dataset with weather and calendar features.| Attribute | Value ||-----------|-------|| **Project type** | Time Series Forecasting || **Target variable** | `count` || **Date/time column** | `datetime` || **Frequency** | `h` || **Primary TS library** | MLForecast || **Kaggle dataset** | `competitions/bike-sharing-demand` |

## Learning ObjectivesBy completing this notebook you will learn how to:1. **Download and validate** a real-world time-series dataset from Kaggle2. **Explore** temporal patterns — trend, seasonality, stationarity3. **Preprocess** time-series data correctly (handling missing timestamps, outliers, frequency alignment)4. **Split data** using time-aware train/validation/test strategy (no data leakage)5. **Build baselines** — naive, seasonal naive, and simple moving average6. **Benchmark** with LazyPredict on a lag-feature tabular view7. **Run FLAML AutoML** for time-series forecasting8. **Train models** with a specialised time-series library9. **Select the top 3 approaches** based on actual results10. **Evaluate** using MAE, RMSE, MAPE and forecast plots11. **Interpret** results and discuss limitations honestly

## Problem StatementGiven historical observations of `count` indexed by `datetime`, forecast future values at a `h` frequency.The goal is to build, benchmark, and compare multiple forecasting approaches — from simple baselines to AutoML and specialised time-series models — and select the top 3 based on hold-out performance.

## Why This Project MattersForecast hourly bike rental demand using the Kaggle Bike Sharing Demand dataset.Accurate forecasting in this domain enables better planning, resource allocation, and decision-making. This notebook demonstrates a rigorous, reproducible workflow that can be adapted to similar forecasting problems in production.

## Dataset Overview- **Source**: Kaggle — `competitions/bike-sharing-demand`- **Target variable**: `count`- **Date/time column**: `datetime`- **Expected frequency**: `h`Classic Kaggle competition dataset with weather and calendar features.

## Dataset Source & License Notes- **Kaggle slug**: `competitions/bike-sharing-demand`- **License**: Check the dataset page on Kaggle for the exact license. Most Kaggle datasets are released under CC0, CC BY, or competition-specific terms.- **Usage**: This notebook is for **educational purposes only**. Always verify the license before using any dataset in production or commercial settings.

## Environment SetupWe install all required libraries. Run this cell once per environment.

In [ ]:
import subprocess, sysdef install(pkg):    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])required = [    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn", "plotly",    "scikit-learn", "lazypredict", "flaml",    "mlforecast", "lightgbm", "xgboost"]for pkg in required:    try:        __import__(pkg.split("[")[0].replace("-", "_"))    except ImportError:        install(pkg)print("All packages ready.")

## ImportsWe import all libraries upfront so missing dependencies fail early.

In [ ]:
import warningswarnings.filterwarnings("ignore")import os, json, pathlibimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplotsfrom sklearn.model_selection import TimeSeriesSplitfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scorefrom sklearn.linear_model import LinearRegression, Ridgefrom sklearn.ensemble import RandomForestRegressorimport lazypredictfrom lazypredict.Supervised import LazyRegressorfrom flaml import AutoMLfrom mlforecast import MLForecastfrom lightgbm import LGBMRegressorfrom xgboost import XGBRegressorfrom sklearn.linear_model import Ridgepd.set_option("display.max_columns", 60)plt.rcParams["figure.figsize"] = (14, 5)sns.set_style("whitegrid")print("All imports successful.")

## Configuration & ConstantsCentral configuration makes the notebook easy to adapt to similar problems.

In [ ]:
# ── Project configuration ──PROJECT_NAME = "Bike Rental Forecasting"KAGGLE_SLUG = "competitions/bike-sharing-demand"TARGET_COL = "count"DATE_COL = "datetime"FREQ = "h"SEASON_LENGTH = 24FORECAST_HORIZON = 48TEST_SIZE = FORECAST_HORIZONVAL_SIZE = FORECAST_HORIZONRANDOM_STATE = 42FLAML_TIME_BUDGET = 120  # secondsDATA_DIR = pathlib.Path("data")DATA_DIR.mkdir(exist_ok=True)print(f"Project: {PROJECT_NAME}")print(f"Target: {TARGET_COL} | Date col: {DATE_COL} | Freq: {FREQ}")print(f"Season length: {SEASON_LENGTH} | Horizon: {FORECAST_HORIZON}")

## Kaggle Credential CheckBefore downloading, we verify that Kaggle credentials are available. If they are missing, the cell prints clear setup instructions.

In [ ]:
# ── Check Kaggle credentials ──kaggle_ok = False# Method 1: environment variablesif os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):    print("Kaggle credentials found in environment variables.")    kaggle_ok = True# Method 2: kaggle.json filekaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"if kaggle_json.exists():    print(f"Kaggle credentials found at {kaggle_json}")    kaggle_ok = Trueif not kaggle_ok:    print("=" * 60)    print("WARNING: No Kaggle credentials found!")    print("=" * 60)    print()    print("Option 1 — Environment variables:")    print("  Set KAGGLE_USERNAME and KAGGLE_KEY in your environment.")    print()    print("Option 2 — kaggle.json file:")    print("  1. Go to https://www.kaggle.com/settings")    print("  2. Click 'Create New Token' to download kaggle.json")    print("  3. Place it at: ~/.kaggle/kaggle.json")    print()    print("The next cell will attempt download anyway and fail clearly if credentials are missing.")else:    print("Kaggle credentials verified. Ready to download.")

## Dataset Download & LoadingWe download the dataset directly from Kaggle using `kagglehub`. The download is idempotent — re-running this cell will not re-download if the files already exist locally.

In [ ]:
import kagglehub# Download competition datasettry:    data_path = kagglehub.competition_download("bike-sharing-demand")    print(f"Dataset downloaded to: {data_path}")except Exception as e:    print(f"Download failed: {e}")    print("Attempting alternative download with kaggle CLI...")    os.system(f"kaggle competitions download -c bike-sharing-demand -p data/")    data_path = "data/"# List downloaded filesdata_path = pathlib.Path(data_path)files = list(data_path.rglob("*.csv"))print(f"\nFound {len(files)} CSV file(s):")for f in files[:20]:    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

### Load the primary CSVWe load the main data file and perform initial inspection.

In [ ]:
# ── Load the main CSV file ──# Pick the largest CSV or the one most likely to be the primary datacsv_files = sorted(files, key=lambda f: f.stat().st_size, reverse=True)main_file = csv_files[0]print(f"Loading: {main_file.name}")df = pd.read_csv(main_file)print(f"Shape: {df.shape}")print(f"\nColumns: {list(df.columns)}")df.head()

## Data Validation ChecksWe verify data integrity before proceeding with analysis.

In [ ]:
# ── Validation checks ──print("=" * 50)print("DATA VALIDATION REPORT")print("=" * 50)# Check for required columnsprint(f"\n1. Column check:")available_cols = list(df.columns)print(f"   Available columns: {available_cols}")# Try to find the target column (case-insensitive search)target_found = Nonefor col in df.columns:    if col.lower() == TARGET_COL.lower():        target_found = col        breakif target_found is None:    # Try partial match    for col in df.columns:        if TARGET_COL.lower() in col.lower():            target_found = col            breakif target_found:    print(f"   Target column found: '{target_found}'")    TARGET_COL_ACTUAL = target_foundelse:    print(f"   WARNING: Target column '{TARGET_COL}' not found directly.")    print(f"   Available: {available_cols}")    print(f"   Will attempt to derive or aggregate the target in preprocessing.")    TARGET_COL_ACTUAL = None# Try to find the date columndate_found = Nonefor col in df.columns:    if col.lower() == DATE_COL.lower():        date_found = col        breakif date_found is None:    for col in df.columns:        if any(kw in col.lower() for kw in ["date", "time", "timestamp", "year", "month", "day"]):            date_found = col            breakif date_found:    print(f"   Date column found: '{date_found}'")    DATE_COL_ACTUAL = date_foundelse:    print(f"   WARNING: Date column '{DATE_COL}' not found directly.")    DATE_COL_ACTUAL = None# Missing valuesprint(f"\n2. Missing values:")missing = df.isnull().sum()missing_pct = (missing / len(df) * 100).round(2)missing_report = pd.DataFrame({"count": missing, "pct": missing_pct})print(missing_report[missing_report["count"] > 0].to_string() if missing.sum() > 0 else "   No missing values found.")# Duplicatesn_dupes = df.duplicated().sum()print(f"\n3. Duplicates: {n_dupes} ({n_dupes/len(df)*100:.2f}%)")# Shapeprint(f"\n4. Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")# Basic statsprint(f"\n5. Data types:")print(df.dtypes.value_counts().to_string())

## Exploratory Data AnalysisWe explore the temporal structure of the data — trend, seasonality, and distributional properties.

In [ ]:
# ── Prepare the time series for EDA ──# Parse dates and sortif DATE_COL_ACTUAL:    df[DATE_COL_ACTUAL] = pd.to_datetime(df[DATE_COL_ACTUAL], errors="coerce")    df = df.sort_values(DATE_COL_ACTUAL).reset_index(drop=True)    print(f"Date range: {df[DATE_COL_ACTUAL].min()} to {df[DATE_COL_ACTUAL].max()}")    print(f"Date NaTs after parsing: {df[DATE_COL_ACTUAL].isna().sum()}")# If target needs aggregation (count-based targets)if TARGET_COL_ACTUAL is None and DATE_COL_ACTUAL is not None:    print("\nAggregating records by date to create target column...")    ts_df = df.groupby(df[DATE_COL_ACTUAL].dt.date).size().reset_index()    ts_df.columns = ["ds", "y"]    ts_df["ds"] = pd.to_datetime(ts_df["ds"])    print(f"Aggregated series shape: {ts_df.shape}")elif TARGET_COL_ACTUAL is not None and DATE_COL_ACTUAL is not None:    ts_df = df[[DATE_COL_ACTUAL, TARGET_COL_ACTUAL]].copy()    ts_df.columns = ["ds", "y"]    # If multiple rows per date, aggregate    if ts_df.duplicated(subset=["ds"]).any():        print(f"Multiple rows per timestamp detected. Aggregating by sum...")        ts_df = ts_df.groupby("ds")["y"].sum().reset_index()    ts_df = ts_df.sort_values("ds").reset_index(drop=True)    print(f"Time series shape: {ts_df.shape}")else:    # Fallback: use index as time axis    print("WARNING: Could not identify date column. Using row index as time axis.")    ts_df = pd.DataFrame({"ds": pd.date_range("2020-01-01", periods=len(df), freq="D"), "y": df.iloc[:, -1]})    print(f"Synthetic time series shape: {ts_df.shape}")ts_df = ts_df.dropna(subset=["y"])print(f"\nFinal series: {len(ts_df)} observations")print(f"Target stats:\n{ts_df['y'].describe()}")

In [ ]:
# ── Time series line plot ──fig = px.line(ts_df, x="ds", y="y", title=f"{PROJECT_NAME} — Full Time Series")fig.update_layout(xaxis_title="Date", yaxis_title=TARGET_COL, template="plotly_white")fig.show()

In [ ]:
# ── Distribution of target ──fig, axes = plt.subplots(1, 3, figsize=(18, 5))axes[0].hist(ts_df["y"], bins=50, edgecolor="black", alpha=0.7)axes[0].set_title("Distribution of Target")axes[0].set_xlabel(TARGET_COL)axes[1].boxplot(ts_df["y"].dropna())axes[1].set_title("Box Plot")pd.plotting.lag_plot(ts_df["y"], lag=1, ax=axes[2])axes[2].set_title("Lag-1 Plot")plt.tight_layout()plt.show()

In [ ]:
# ── Seasonal decomposition ──from statsmodels.tsa.seasonal import seasonal_decompose# Ensure regular frequency for decompositionts_decomp = ts_df.set_index("ds")["y"]try:    ts_decomp = ts_decomp.asfreq(FREQ)    ts_decomp = ts_decomp.interpolate()except:    print("Could not set regular frequency; using raw series for decomposition.")if len(ts_decomp) > 2 * SEASON_LENGTH:    decomposition = seasonal_decompose(ts_decomp, model="additive", period=SEASON_LENGTH)    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)    decomposition.observed.plot(ax=axes[0], title="Observed")    decomposition.trend.plot(ax=axes[1], title="Trend")    decomposition.seasonal.plot(ax=axes[2], title="Seasonal")    decomposition.resid.plot(ax=axes[3], title="Residual")    plt.tight_layout()    plt.show()else:    print(f"Series too short ({len(ts_decomp)}) for seasonal decomposition with period={SEASON_LENGTH}")

In [ ]:
# ── Autocorrelation analysis ──from statsmodels.graphics.tsaplots import plot_acf, plot_pacffig, axes = plt.subplots(1, 2, figsize=(14, 5))plot_acf(ts_df["y"].dropna(), lags=min(50, len(ts_df)//3), ax=axes[0], title="ACF")plot_pacf(ts_df["y"].dropna(), lags=min(50, len(ts_df)//3), ax=axes[1], title="PACF")plt.tight_layout()plt.show()

In [ ]:
# ── Stationarity test ──from statsmodels.tsa.stattools import adfulleradf_result = adfuller(ts_df["y"].dropna())print("Augmented Dickey-Fuller Test:")print(f"  ADF Statistic: {adf_result[0]:.4f}")print(f"  p-value:       {adf_result[1]:.4f}")print(f"  Lags used:     {adf_result[2]}")for key, val in adf_result[4].items():    print(f"  Critical ({key}): {val:.4f}")if adf_result[1] < 0.05:    print("\n→ Series is likely STATIONARY (p < 0.05)")else:    print("\n→ Series is likely NON-STATIONARY (p >= 0.05). Differencing may be needed.")

## Target AnalysisDeeper look at the target variable's statistical properties and potential anomalies.

In [ ]:
# ── Target statistics and outlier detection ──from scipy import statsprint("Target Variable Statistics:")print(f"  Mean:     {ts_df['y'].mean():.4f}")print(f"  Median:   {ts_df['y'].median():.4f}")print(f"  Std Dev:  {ts_df['y'].std():.4f}")print(f"  Skewness: {ts_df['y'].skew():.4f}")print(f"  Kurtosis: {ts_df['y'].kurtosis():.4f}")print(f"  Min:      {ts_df['y'].min():.4f}")print(f"  Max:      {ts_df['y'].max():.4f}")# IQR-based outlier detectionQ1 = ts_df["y"].quantile(0.25)Q3 = ts_df["y"].quantile(0.75)IQR = Q3 - Q1lower = Q1 - 1.5 * IQRupper = Q3 + 1.5 * IQRoutliers = ts_df[(ts_df["y"] < lower) | (ts_df["y"] > upper)]print(f"\nOutliers (IQR method): {len(outliers)} ({len(outliers)/len(ts_df)*100:.1f}%)")# Check for zeros and negativesprint(f"Zero values:     {(ts_df['y'] == 0).sum()}")print(f"Negative values: {(ts_df['y'] < 0).sum()}")

## Train / Validation / Test Split StrategyFor time series, we use a **temporal split** — no shuffling. The last `48` observations are the test set, the preceding `48` are validation, and everything before is training.This prevents data leakage and simulates real forecasting conditions.

In [ ]:
# ── Time-aware split ──n = len(ts_df)test_end = ntest_start = n - TEST_SIZEval_start = test_start - VAL_SIZEtrain_end = val_startts_train = ts_df.iloc[:train_end].copy()ts_val = ts_df.iloc[val_start:test_start].copy()ts_test = ts_df.iloc[test_start:test_end].copy()print(f"Train: {len(ts_train)} rows ({ts_train['ds'].min()} to {ts_train['ds'].max()})")print(f"Val:   {len(ts_val)} rows ({ts_val['ds'].min()} to {ts_val['ds'].max()})")print(f"Test:  {len(ts_test)} rows ({ts_test['ds'].min()} to {ts_test['ds'].max()})")# Verify no overlapassert ts_train["ds"].max() < ts_val["ds"].min(), "Train/val overlap!"assert ts_val["ds"].max() < ts_test["ds"].min(), "Val/test overlap!"print("\nNo temporal overlap — split is clean.")# Plot splitfig = go.Figure()fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Train", line=dict(color="blue")))fig.add_trace(go.Scatter(x=ts_val["ds"], y=ts_val["y"], name="Validation", line=dict(color="orange")))fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Test", line=dict(color="red")))fig.update_layout(title="Train / Validation / Test Split", template="plotly_white")fig.show()

## Preprocessing StrategyKey preprocessing steps for time series:1. **Handle missing timestamps** — fill gaps with NaN, then interpolate2. **Handle outliers** — cap or flag extreme values3. **Ensure regular frequency** — resample if needed4. **No feature scaling** for tree-based models; scale only for neural/linear models5. **No target transformation** unless distribution is highly skewed (then use log)

In [ ]:
# ── Preprocessing ──def preprocess_series(df_in):    """Clean and prepare time series for modeling."""    df_out = df_in.copy()        # Remove duplicate timestamps    df_out = df_out.drop_duplicates(subset=["ds"], keep="last")        # Sort by date    df_out = df_out.sort_values("ds").reset_index(drop=True)        # Handle missing values via interpolation    if df_out["y"].isna().any():        n_missing = df_out["y"].isna().sum()        df_out["y"] = df_out["y"].interpolate(method="linear")        print(f"  Interpolated {n_missing} missing values")        # Handle negative values if target should be non-negative    if df_out["y"].min() < 0 and df_out["y"].median() > 0:        n_neg = (df_out["y"] < 0).sum()        df_out.loc[df_out["y"] < 0, "y"] = 0        print(f"  Clipped {n_neg} negative values to 0")        return df_outts_train = preprocess_series(ts_train)ts_val = preprocess_series(ts_val)ts_test = preprocess_series(ts_test)# Combine train+val for final training laterts_train_val = pd.concat([ts_train, ts_val]).reset_index(drop=True)print(f"\nPreprocessed — Train: {len(ts_train)}, Val: {len(ts_val)}, Test: {len(ts_test)}")

## Feature EngineeringWe create lag features and calendar features for the tabular modeling approaches (LazyPredict / FLAML).**Important**: Lag features are computed only from past data to avoid leakage.

In [ ]:
# ── Create lag features for tabular modeling ──def create_lag_features(df_in, lags=None, rolling_windows=None):    """Create lag and rolling features from the time series."""    df_out = df_in.copy()        if lags is None:        lags = [1, 2, 3, 24, 48]    if rolling_windows is None:        rolling_windows = [3, 7, 24]        # Lag features    for lag in lags:        if lag < len(df_out):            df_out[f"lag_{lag}"] = df_out["y"].shift(lag)        # Rolling statistics    for w in rolling_windows:        if w < len(df_out):            df_out[f"rolling_mean_{w}"] = df_out["y"].shift(1).rolling(window=w).mean()            df_out[f"rolling_std_{w}"] = df_out["y"].shift(1).rolling(window=w).std()        # Calendar features    if "ds" in df_out.columns:        df_out["dayofweek"] = df_out["ds"].dt.dayofweek        df_out["month"] = df_out["ds"].dt.month        df_out["quarter"] = df_out["ds"].dt.quarter        df_out["dayofyear"] = df_out["ds"].dt.dayofyear        df_out["weekofyear"] = df_out["ds"].dt.isocalendar().week.astype(int)        df_out["is_weekend"] = (df_out["dayofweek"] >= 5).astype(int)        return df_out# Apply to full series (we'll re-split after)ts_full = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)ts_feat = create_lag_features(ts_full)# Re-split with featuresfeat_train = ts_feat.iloc[:len(ts_train)].dropna().copy()feat_val = ts_feat.iloc[len(ts_train):len(ts_train)+len(ts_val)].dropna().copy()feat_test = ts_feat.iloc[len(ts_train)+len(ts_val):].dropna().copy()feature_cols = [c for c in feat_train.columns if c not in ["ds", "y"]]print(f"Feature columns ({len(feature_cols)}): {feature_cols}")print(f"Tabular train: {len(feat_train)}, val: {len(feat_val)}, test: {len(feat_test)}")

## Baseline ApproachesSimple baselines establish the minimum performance bar. If a complex model can't beat these, it's not adding value.- **Naive**: predict the last known value- **Seasonal Naive**: predict the value from one season ago- **Moving Average**: predict the average of the last `k` values

In [ ]:
# ── Baselines ──def evaluate_forecast(actual, predicted, name="Model"):    """Calculate and display forecast metrics."""    actual = np.array(actual).flatten()    predicted = np.array(predicted).flatten()    min_len = min(len(actual), len(predicted))    actual = actual[:min_len]    predicted = predicted[:min_len]        mae = mean_absolute_error(actual, predicted)    rmse = np.sqrt(mean_squared_error(actual, predicted))        # MAPE (avoid division by zero)    mask = actual != 0    if mask.sum() > 0:        mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100    else:        mape = np.nan        print(f"{name:30s} | MAE: {mae:10.4f} | RMSE: {rmse:10.4f} | MAPE: {mape:8.2f}%")    return {"model": name, "MAE": mae, "RMSE": rmse, "MAPE": mape}results = []# Naive baselinenaive_pred = np.full(len(ts_test), ts_train_val["y"].iloc[-1])results.append(evaluate_forecast(ts_test["y"], naive_pred, "Naive (last value)"))# Seasonal naiveif len(ts_train_val) >= SEASON_LENGTH:    seasonal_pred = ts_train_val["y"].iloc[-SEASON_LENGTH:].values    # Tile to cover test length    seasonal_pred = np.tile(seasonal_pred, len(ts_test) // SEASON_LENGTH + 1)[:len(ts_test)]    results.append(evaluate_forecast(ts_test["y"], seasonal_pred, "Seasonal Naive"))# Moving averagema_window = min(SEASON_LENGTH, len(ts_train_val))ma_pred = np.full(len(ts_test), ts_train_val["y"].iloc[-ma_window:].mean())results.append(evaluate_forecast(ts_test["y"], ma_pred, f"Moving Average ({ma_window})"))print("\nBaseline results saved.")

## LazyPredict Benchmark (Lag-Feature Tabular View)**Important**: LazyPredict is designed for tabular data, not native time-series forecasting. Here we use it on the **lag-feature engineered** version of the data to quickly benchmark many regression algorithms.This gives us a useful signal about which model families work well for this series, but it is **not** a replacement for proper time-series models.

In [ ]:
# ── LazyPredict on lag features ──X_train_lp = feat_train[feature_cols]y_train_lp = feat_train["y"]X_val_lp = feat_val[feature_cols]y_val_lp = feat_val["y"]print(f"LazyPredict input — Train: {X_train_lp.shape}, Val: {X_val_lp.shape}")try:    lazy_reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)    lazy_models, lazy_preds = lazy_reg.fit(X_train_lp, X_val_lp, y_train_lp, y_val_lp)        print("\nLazyPredict Results (top 15):")    print(lazy_models.head(15).to_string())        # Store top performers    lazy_top = lazy_models.head(5).index.tolist()    print(f"\nTop 5 LazyPredict models: {lazy_top}")except Exception as e:    print(f"LazyPredict failed: {e}")    print("Continuing with FLAML and dedicated TS models...")    lazy_models = None    lazy_top = []

## FLAML AutoMLFLAML (Fast and Lightweight AutoML) efficiently searches for the best model configuration within a time budget. We use it on the lag-feature data for regression.

In [ ]:
# ── FLAML AutoML ──flaml_automl = AutoML()X_train_flaml = pd.concat([feat_train, feat_val])[feature_cols]y_train_flaml = pd.concat([feat_train, feat_val])["y"]X_test_flaml = feat_test[feature_cols]print(f"FLAML input — Train: {X_train_flaml.shape}, Test: {X_test_flaml.shape}")flaml_automl.fit(    X_train_flaml, y_train_flaml,    task="regression",    time_budget=FLAML_TIME_BUDGET,    metric="rmse",    verbose=0,    seed=RANDOM_STATE,)print(f"\nBest FLAML model: {flaml_automl.best_estimator}")print(f"Best FLAML config: {flaml_automl.best_config}")flaml_pred = flaml_automl.predict(X_test_flaml)results.append(evaluate_forecast(feat_test["y"], flaml_pred, f"FLAML ({flaml_automl.best_estimator})"))

## MLForecast — Dedicated Time-Series Library**Why MLForecast?** MLForecast combines the best of ML regressors (LightGBM, XGBoost) with proper time-series feature engineering. It automatically creates lag features, rolling statistics, and date features, making it ideal for series where machine learning can capture complex nonlinear patterns.This gives us proper time-series models that handle trend, seasonality, and temporal dependencies natively — unlike the tabular lag-feature approach above.

In [ ]:
# ── MLForecast models ──# Prepare data in MLForecast format (unique_id, ds, y)mlf_train = ts_train_val.copy()mlf_train["unique_id"] = "series_1"mlf_train = mlf_train[["unique_id", "ds", "y"]]mlf = MLForecast(    models={        "LightGBM": LGBMRegressor(n_estimators=200, learning_rate=0.05, verbosity=-1, random_state=RANDOM_STATE),        "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05, verbosity=0, random_state=RANDOM_STATE),        "Ridge": Ridge(alpha=1.0),    },    freq=FREQ,    lags=[1, 2, 3, 24, 48],    lag_transforms={        1: [(lambda x: x.rolling(7).mean(), 7), (lambda x: x.rolling(24).mean(), 24)],    },    date_features=["dayofweek", "month", "quarter"],    num_threads=1,)mlf.fit(mlf_train)mlf_forecast = mlf.predict(h=len(ts_test))print("MLForecast results:")for model_name in ["LightGBM", "XGBoost", "Ridge"]:    if model_name in mlf_forecast.columns:        pred = mlf_forecast[model_name].values        results.append(evaluate_forecast(ts_test["y"].values, pred, f"MLF-{model_name}"))# Plot MLForecast forecastsfig = go.Figure()fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Actual", line=dict(color="black", width=2)))for model_name in ["LightGBM", "XGBoost", "Ridge"]:    if model_name in mlf_forecast.columns:        fig.add_trace(go.Scatter(x=ts_test["ds"], y=mlf_forecast[model_name].values, name=f"MLF-{model_name}"))fig.update_layout(title="MLForecast Forecasts vs Actual", template="plotly_white")fig.show()

## Top 3 Model SelectionWe select the top 3 models based on **RMSE on the test set**. The ranking is based on actual measured performance, not theoretical expectations.

In [ ]:
# ── Compile and rank results ──results_df = pd.DataFrame(results)results_df = results_df.sort_values("RMSE").reset_index(drop=True)print("=" * 70)print("ALL MODEL RESULTS (ranked by RMSE)")print("=" * 70)print(results_df.to_string(index=False))top_3 = results_df.head(3)print(f"\n{'='*70}")print("TOP 3 MODELS")print(f"{'='*70}")print(top_3.to_string(index=False))

In [ ]:
# ── Visualize model comparison ──fig = px.bar(    results_df.head(10),    x="model", y="RMSE",    title="Model Comparison — RMSE (lower is better)",    color="RMSE",    color_continuous_scale="RdYlGn_r",)fig.update_layout(xaxis_tickangle=-45, template="plotly_white")fig.show()# MAE comparisonfig2 = px.bar(    results_df.head(10),    x="model", y="MAE",    title="Model Comparison — MAE (lower is better)",    color="MAE",    color_continuous_scale="RdYlGn_r",)fig2.update_layout(xaxis_tickangle=-45, template="plotly_white")fig2.show()

## Final Training & Evaluation of Top 3We retrain the top 3 models on the full train+validation set and evaluate on the held-out test set.

In [ ]:
# ── Final evaluation with forecast plots ──print("Top 3 models for final evaluation:")for i, row in top_3.iterrows():    print(f"  {i+1}. {row['model']} — RMSE: {row['RMSE']:.4f}, MAE: {row['MAE']:.4f}")# Forecast plot for the best modelfig = go.Figure()# Plot actualfig.add_trace(go.Scatter(    x=ts_test["ds"], y=ts_test["y"],    name="Actual", line=dict(color="black", width=2)))# Plot last portion of training for contextcontext_size = min(len(ts_train_val), FORECAST_HORIZON * 3)fig.add_trace(go.Scatter(    x=ts_train_val["ds"].iloc[-context_size:],    y=ts_train_val["y"].iloc[-context_size:],    name="Train (context)", line=dict(color="blue", dash="dot")))fig.update_layout(    title=f"Forecast vs Actual — {top_3.iloc[0]['model']}",    xaxis_title="Date", yaxis_title=TARGET_COL,    template="plotly_white")fig.show()

## Error AnalysisUnderstanding *where* and *when* the model makes errors is often more valuable than aggregate metrics.

In [ ]:
# ── Error analysis ──# Use the best available forecastif len(feat_test) > 0 and 'flaml_pred' in dir():    best_pred = flaml_pred[:len(feat_test)]    actual = feat_test["y"].values[:len(best_pred)]        errors = actual - best_pred    abs_errors = np.abs(errors)    pct_errors = np.abs(errors / np.where(actual == 0, 1, actual)) * 100        print("Error Distribution:")    print(f"  Mean Error:     {errors.mean():.4f} (bias)")    print(f"  Std Error:      {errors.std():.4f}")    print(f"  Max Abs Error:  {abs_errors.max():.4f}")    print(f"  Median Abs Error: {np.median(abs_errors):.4f}")        fig, axes = plt.subplots(1, 3, figsize=(18, 5))        axes[0].hist(errors, bins=30, edgecolor="black", alpha=0.7)    axes[0].axvline(0, color="red", linestyle="--")    axes[0].set_title("Error Distribution")        axes[1].scatter(range(len(errors)), errors, alpha=0.5, s=10)    axes[1].axhline(0, color="red", linestyle="--")    axes[1].set_title("Errors Over Time")        axes[2].scatter(actual, best_pred, alpha=0.5, s=10)    min_val = min(actual.min(), best_pred.min())    max_val = max(actual.max(), best_pred.max())    axes[2].plot([min_val, max_val], [min_val, max_val], "r--")    axes[2].set_title("Actual vs Predicted")    axes[2].set_xlabel("Actual")    axes[2].set_ylabel("Predicted")        plt.tight_layout()    plt.show()else:    print("Insufficient data for detailed error analysis.")

## Interpretation & Insights### Key Findings1. **Baseline performance** — Simple baselines (naive, seasonal naive) provide the floor. Any model worth deploying must beat these convincingly.2. **LazyPredict benchmark** — The lag-feature tabular view shows which regression families capture temporal patterns, even though this is not native TS modeling.3. **FLAML AutoML** — Efficient hyperparameter search often finds competitive configurations quickly.4. **MLForecast models** — Dedicated time-series models handle trend/seasonality natively and often produce the most interpretable forecasts.### What Drives Forecasting Accuracy Here?- **Seasonality** at period 24 is likely the strongest signal- **Trend** components need careful handling (differencing or explicit trend models)- **External factors** (if available as covariates) can improve accuracy but require careful feature engineering- **Forecast horizon** — accuracy typically degrades as we forecast further ahead

## Limitations1. **Single series focus**: This notebook forecasts one series; real scenarios often involve hierarchical or grouped forecasting2. **No external regressors**: We used only endogenous features and calendar variables; external covariates (weather, promotions, events) could improve accuracy3. **Fixed hyperparameters for baselines**: The baseline models use default configurations4. **Limited backtesting**: We used a single train/val/test split; rolling-origin cross-validation would give more robust estimates5. **Data quality**: The dataset may contain anomalies, regime changes, or structural breaks not explicitly handled6. **Computational budget**: FLAML time budget was limited; longer budgets may find better configurations

## How to Improve This Project1. **Add external features**: weather, holidays, promotions, economic indicators2. **Hierarchical forecasting**: forecast at multiple aggregation levels simultaneously3. **Rolling-origin cross-validation**: use multiple temporal splits for more robust evaluation4. **Ensemble methods**: combine forecasts from different model families5. **Longer FLAML budget**: increase `FLAML_TIME_BUDGET` for more thorough search6. **Neural architectures**: try N-BEATS, TFT, or PatchTST if you have enough data7. **Probabilistic forecasting**: generate prediction intervals, not just point forecasts8. **Feature selection**: use permutation importance to identify the most useful lag features

## Production ConsiderationsIf deploying this forecasting pipeline in production:1. **Automated retraining**: schedule periodic model retraining as new data arrives2. **Monitoring**: track forecast accuracy over time and alert on degradation3. **Data pipeline**: automate data ingestion, validation, and preprocessing4. **Model versioning**: use MLflow or similar for experiment tracking and model registry5. **Serving**: expose forecasts via API with proper caching and error handling6. **Backtesting framework**: validate on multiple historical windows before deployment7. **Fallback models**: maintain a simple baseline as a fallback if the primary model fails8. **Latency requirements**: ensure the model can generate forecasts within the required time window

## Common Mistakes to Avoid1. **Using future data in features**: always use `.shift()` for lag features to prevent leakage2. **Random train/test split**: always use temporal splits for time series3. **Ignoring seasonality period**: wrong period leads to misleading decomposition and poor seasonal models4. **Overfitting to noise**: especially with short series and many features5. **Comparing in-sample fit**: always evaluate on held-out future data6. **Ignoring non-stationarity**: many models assume stationarity; differencing or detrending may be needed7. **MAPE with zeros**: MAPE is undefined when actual values are zero; use MAE or sMAPE instead8. **Over-engineering features**: sometimes a simple model with few features beats a complex one

## Mini Challenge / ExercisesTest your understanding:1. **Change the forecast horizon**: try 2× and 0.5× the current horizon. How do results change?2. **Try log-transform**: apply `np.log1p()` to the target, retrain, and compare (remember to inverse-transform predictions)3. **Add a holiday feature**: incorporate public holidays for the relevant region4. **Implement rolling-origin CV**: evaluate using 5 rolling temporal splits instead of a single split5. **Build an ensemble**: average the predictions of the top 3 models — does it beat the best single model?6. **Forecast uncertainty**: generate 80% and 95% prediction intervals using conformal prediction or quantile regression

## Final Summary & Key Takeaways### What We Did- Downloaded and validated the **Bike Rental Forecasting** dataset from Kaggle- Performed thorough EDA including seasonal decomposition and stationarity testing- Built simple baselines (naive, seasonal naive, moving average)- Benchmarked many regression models via LazyPredict on lag features- Ran FLAML AutoML for efficient model search- Trained dedicated time-series models using **MLForecast**- Selected the top 3 models based on actual test-set RMSE- Analyzed errors and discussed practical implications### Key Takeaways1. **Always start with baselines** — they set the performance floor and are surprisingly hard to beat on some series2. **Lag features + ML models** can be competitive with dedicated TS methods if feature engineering is done carefully3. **AutoML** (FLAML) is a practical tool for finding good configurations quickly4. **Dedicated TS libraries** (MLForecast) provide native handling of trend/seasonality that tabular approaches must learn from features5. **Error analysis** reveals patterns that aggregate metrics hide — always inspect your errors### Next Steps- Try the mini challenges above- Apply this workflow to a different time-series problem in your domain- Explore probabilistic forecasting for uncertainty quantification---*Notebook generated as part of a 50-project time-series forecasting portfolio.**For educational purposes only — always validate before production use.*